# Adaptive Portfolio Positioning under Market Volatility

**Two-factor TVTP Markov-Switching GJR-GARCH**  
Daily regime detection · Soft position mapping · 5-day minimum holding period

This notebook reproduces the **main results** of the paper.  
Primary specification: lagged log(VIX) + VIX term-structure inversion as transition drivers.

**Frozen execution parameters** (locked):
- Mode: soft threshold
- τ = 0.65
- W_STRESS = 0.50
- Min holding period = 5 trading days

Evaluation: strict expanding-window out-of-sample (first estimation at day 1500, re-estimated every ~130 days).


## 1. Environment

In [ ]:
# Environment Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.special import logsumexp
from numba import njit
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.figsize": (14, 6),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("Environment ready. Random seed =", RANDOM_SEED)
print("Hybrid: daily detection | MIN_HOLD = 5 trading days | TWO-FACTOR TVTP")


## 2. Data

In [ ]:
# Data Download — DAILY panel (hybrid, two-factor ready)
# Features (lagged >= 1 day, no look-ahead):
#   log_VIX_lagged, Inversion_lagged
# Panel kept under name `weekly` for downstream compatibility.

import yfinance as yf

def download_close(ticker, start="2010-01-01", end="2026-07-31"):
    """Robust close extractor (handles MultiIndex from yfinance)."""
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        s = raw["Close"]
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:, 0]
        return s
    return raw["Close"]

spy_d   = download_close("SPY")
vix_d   = download_close("^VIX")
vix3m_d = download_close("^VIX3M")
bil_d   = download_close("BIL")

daily = pd.concat([spy_d, vix_d, vix3m_d, bil_d], axis=1, join="inner")
daily.columns = ["SPY", "VIX", "VIX3M", "BIL"]
daily = daily.dropna()
daily["SPY_return"] = daily["SPY"].pct_change()
daily["BIL_return"] = daily["BIL"].pct_change()
daily["Inversion"]  = daily["VIX"] / daily["VIX3M"]
daily["log_VIX"]    = np.log(daily["VIX"].astype(float).clip(lower=1e-6))

# Strict lags
daily["Inversion_lagged"] = daily["Inversion"].shift(1)
daily["log_VIX_lagged"]   = daily["log_VIX"].shift(1)
daily = daily.dropna()

weekly = daily.copy()
weekly.to_csv("data_daily_hybrid_2f.csv")

print(f"Daily T = {len(weekly)}  |  {weekly.index.min().date()} → {weekly.index.max().date()}")
print(weekly[["SPY_return", "log_VIX_lagged", "Inversion_lagged", "BIL_return"]].describe().round(4))


## 3. Model Specification (Two-factor TVTP)

In [ ]:
# Two-factor TVTP — log(VIX) + Inversion
# z_t (lagged, no look-ahead):
#   z1 = log(VIX)_{t-1}
#   z2 = Inversion_{t-1} = VIX / VIX3M
#
# Transition logits:
#   From Normal (0): stress-entry logit += s_enter · z ,  s_enter = exp(.) >= 0
#   From Stress (1): exit-to-Normal logit += s_exit · z , s_exit free
# Within-regime: GJR-GARCH(1,1); dual ID (mu descending, omega ascending)
# Soft persistence: daily-scale stay targets + Stress-mass target

returns          = weekly["SPY_return"].values.astype(np.float64)
inversion_lagged = weekly["Inversion_lagged"].values.astype(np.float64)
logvix_lagged    = weekly["log_VIX_lagged"].values.astype(np.float64)
T = len(returns)
K = 2
N_FEAT = 2  # [log_VIX, Inversion]
print(f"Two-factor TVTP | T = {T}  |  K = {K}  |  z = [logVIX, Inversion]")

def unpack_parameters(theta):
    """
    theta layout:
      mu[K], log_omega[K], log_alpha[K], gamma[K], logit_beta[K],
      intercept[K*K] raveled,
      log_s_enter[N_FEAT],   # >= 0 after exp
      s_exit[N_FEAT]         # free
    """
    theta = np.asarray(theta, dtype=np.float64)
    mu    = theta[0:K].copy()
    omega = np.exp(theta[K:2*K])
    alpha = np.exp(theta[2*K:3*K])
    gamma = theta[3*K:4*K].copy()
    beta  = 1.0 / (1.0 + np.exp(-theta[4*K:5*K]))

    order = np.argsort(-mu)  # dual ID: descending mean
    mu, omega, alpha, gamma, beta = (mu[order], omega[order], alpha[order],
                                     gamma[order], beta[order])
    base = 5 * K
    inter = theta[base:base + K*K].reshape(K, K)[order, :][:, order]
    base2 = base + K*K
    s_enter = np.exp(theta[base2:base2 + N_FEAT])
    s_exit  = theta[base2 + N_FEAT:base2 + 2*N_FEAT].copy()
    return mu, omega, alpha, gamma, beta, inter, s_enter, s_exit


def transition_matrix(z, trans_intercept, s_enter, s_exit):
    """z: array-like shape (2,)"""
    z = np.asarray(z, dtype=np.float64)
    P = np.zeros((K, K))
    logits0 = trans_intercept[0].copy()
    logits0[1] = logits0[1] + float(s_enter @ z)
    P[0, :] = np.exp(logits0 - logsumexp(logits0))
    logits1 = trans_intercept[1].copy()
    logits1[0] = logits1[0] + float(s_exit @ z)
    P[1, :] = np.exp(logits1 - logsumexp(logits1))
    return P


def build_P_all(z_path, intercept, s_enter, s_exit):
    """z_path: (T, 2)"""
    T_ = z_path.shape[0]
    logits = np.broadcast_to(intercept, (T_, K, K)).copy()
    logits[:, 0, 1] = logits[:, 0, 1] + z_path @ s_enter
    logits[:, 1, 0] = logits[:, 1, 0] + z_path @ s_exit
    m = logits.max(axis=2, keepdims=True)
    e = np.exp(logits - m)
    return e / e.sum(axis=2, keepdims=True)


@njit(cache=False)
def nll_numba(ret, mu, omega, alpha, gamma, beta, P_all):
    T_ = len(ret)
    xi = np.ones(K) / float(K)
    s2 = np.full(K, np.var(ret[:min(50, T_)]))
    ll = 0.0
    for t in range(T_):
        xi_pred = np.zeros(K)
        for j in range(K):
            acc = 0.0
            for i in range(K):
                acc += P_all[t, i, j] * xi[i]
            xi_pred[j] = acc
        dens = np.empty(K)
        s2n  = np.empty(K)
        for k in range(K):
            if t > 0:
                eps  = ret[t - 1] - mu[k]
                Ineg = 1.0 if eps < 0.0 else 0.0
            else:
                eps, Ineg = 0.0, 0.0
            s2n[k] = omega[k] + (alpha[k] + gamma[k] * Ineg) * (eps * eps) + beta[k] * s2[k]
            if s2n[k] < 1e-12:
                s2n[k] = 1e-12
            r = ret[t] - mu[k]
            dens[k] = np.exp(-0.5 * np.log(2.0 * np.pi * s2n[k]) - 0.5 * (r * r) / s2n[k])
            if dens[k] < 1e-300:
                dens[k] = 1e-300
        joint_sum = 0.0
        for k in range(K):
            joint_sum += xi_pred[k] * dens[k]
        if joint_sum < 1e-300:
            return 1e12
        for k in range(K):
            xi[k] = xi_pred[k] * dens[k] / joint_sum
            s2[k] = s2n[k]
        ll += np.log(joint_sum)
    return -ll


def stationary_dist(P):
    p01 = float(P[0, 1])
    p10 = float(P[1, 0])
    denom = p01 + p10
    if denom < 1e-12:
        return np.array([0.5, 0.5])
    pi0 = p10 / denom
    return np.array([pi0, 1.0 - pi0])


def negative_log_likelihood(theta):
    mu, omega, alpha, gamma, beta, t_int, s_enter, s_exit = unpack_parameters(theta)
    if not (omega[0] < omega[1]):
        return 1e12 + 1e6 * max(0.0, omega[0] - omega[1])

    z_path = np.column_stack([logvix_lagged, inversion_lagged])
    P_all = build_P_all(z_path, t_int, s_enter, s_exit)
    nll = float(nll_numba(returns, mu, omega, alpha, gamma, beta, P_all))

    P_avg = P_all.mean(axis=0)
    stay0 = float(P_avg[0, 0])
    stay1 = float(P_avg[1, 1])
    pi_stat = stationary_dist(P_avg)
    stress_mass = float(pi_stat[1])

    TARGET_STAY0 = 0.97
    TARGET_STAY1 = 0.90
    TARGET_STRESS_MASS = 0.25
    STRESS_MASS_BAND = 0.05
    PEN_STAY, PEN_MASS = 200.0, 150.0

    pen_stay = PEN_STAY * (
        max(0.0, TARGET_STAY0 - stay0) ** 2 +
        max(0.0, TARGET_STAY1 - stay1) ** 2
    )
    excess = max(0.0, stress_mass - (TARGET_STRESS_MASS + STRESS_MASS_BAND))
    shortfall = max(0.0, (TARGET_STRESS_MASS - STRESS_MASS_BAND) - stress_mass)
    pen_mass = PEN_MASS * (excess ** 2 + 0.25 * shortfall ** 2)
    return nll + pen_stay + pen_mass


def theta_dim():
    return 5 * K + K * K + 2 * N_FEAT


def random_theta(rng):
    mu_init = np.array([0.0005, -0.0008]) + rng.normal(0.0, 0.0003, K)
    omega_init = np.log(np.array([3e-6, 2e-5]) * (0.5 + rng.random(K)))
    alpha_init = np.log(np.clip(np.array([0.04, 0.08]) * (0.5 + rng.random(K)), 1e-4, 0.40))
    gamma_init = np.array([0.08, 0.20]) + rng.normal(0.0, 0.04, K)
    beta_raw = np.clip(np.array([0.85, 0.75]) + rng.normal(0.0, 0.04, K), 0.55, 0.97)
    beta_init = np.log(beta_raw / (1.0 - beta_raw))
    inter_init = rng.normal(0.0, 0.30, (K, K))
    np.fill_diagonal(inter_init, 2.5 + 0.8 * rng.random(K))
    log_s_enter = rng.normal(-0.5, 0.4, N_FEAT)
    s_exit = rng.normal(0.0, 0.3, N_FEAT)
    return np.concatenate([
        mu_init, omega_init, alpha_init, gamma_init, beta_init,
        inter_init.ravel(), log_s_enter, s_exit,
    ])


_ = negative_log_likelihood(random_theta(np.random.default_rng(0)) * 0.1)
print("Two-factor TVTP ready | dim(theta) =", theta_dim())
print("  s_enter on [log_VIX, Inversion]  >= 0")
print("  s_exit  on [log_VIX, Inversion]  free")


## 4. Parameter Estimation (Multi-start MLE)

In [ ]:
# Multi-start MLE (two-factor TVTP)
from scipy.optimize import minimize
import time

N_STARTS = 16
MAX_ITER = 400
FTOL = 1e-9

print(f"Multi-start MLE (two-factor): {N_STARTS} starts")
print("=" * 72)
best_nll = np.inf
best_theta = None
t0 = time.time()

for s in range(N_STARTS):
    rng = np.random.default_rng(RANDOM_SEED + s * 19)
    theta0 = random_theta(rng)
    t_s = time.time()
    try:
        res = minimize(
            negative_log_likelihood, theta0,
            method="L-BFGS-B",
            options={"maxiter": MAX_ITER, "ftol": FTOL, "disp": False},
        )
        fun = float(res.fun) if np.isfinite(res.fun) else np.inf
    except Exception as e:
        fun = np.inf
        res = None
        print(f"  Start {s+1:02d} exception: {e}")
    dt = time.time() - t_s
    flag = ""
    if fun < best_nll:
        best_nll = fun
        best_theta = res.x.copy() if res is not None else best_theta
        flag = "  <- new best"
    print(f"Start {s+1:02d}: NLL+pen = {fun:12.4f}  ({dt:.1f}s){flag}")

print("=" * 72)
print(f"Best NLL+pen = {best_nll:.4f}  |  elapsed {(time.time()-t0)/60:.1f} min")
assert best_theta is not None, "MLE failed on all starts"

mu, omega, alpha, gamma, beta, trans_intercept, s_enter, s_exit = unpack_parameters(best_theta)
print("\ns_enter [logVIX, Inv] (>=0):", np.round(s_enter, 4))
print("s_exit  [logVIX, Inv] (free):", np.round(s_exit, 4))
print(f"ID check: mu_desc={mu[0]>mu[1]}, omega_asc={omega[0]<omega[1]}")


## 5. Filtering and Regime Diagnostics

In [ ]:
# Filter / smoother + regime diagnostics (two-factor)
mu, omega, alpha, gamma, beta, trans_intercept, s_enter, s_exit = unpack_parameters(best_theta)
z_path = np.column_stack([logvix_lagged, inversion_lagged])

print("=" * 72)
print("Estimated parameters (two-factor, asymmetric persistence)")
print("=" * 72)
names = ["0 Normal Growth", "1 High-Vol/Stress"]
print(f"{'Regime':<22} {'mu':>10} {'omega':>12} {'alpha':>8} {'gamma':>8} {'beta':>8}")
print("-" * 72)
for k in range(K):
    print(f"{names[k]:<22} {mu[k]:10.6f} {omega[k]:12.2e} {alpha[k]:8.4f} "
          f"{gamma[k]:8.4f} {beta[k]:8.4f}")
print(f"\ns_enter (>=0) = {np.round(s_enter, 4)}")
print(f"s_exit        = {np.round(s_exit, 4)}")

xi_filtered = np.zeros((T, K))
sigma2_path = np.zeros((T, K))
xi_prev = np.ones(K) / K
sigma2_prev = np.full(K, np.var(returns[:min(50, T)]))

for t in range(T):
    P = transition_matrix(z_path[t], trans_intercept, s_enter, s_exit)
    xi_pred = P.T @ xi_prev
    if t > 0:
        eps_prev = returns[t-1] - mu
        I_neg = (eps_prev < 0).astype(np.float64)
    else:
        eps_prev = np.zeros(K)
        I_neg = np.zeros(K)
    sigma2 = omega + (alpha + gamma * I_neg) * (eps_prev ** 2) + beta * sigma2_prev
    sigma2 = np.maximum(sigma2, 1e-12)
    sigma2_path[t] = sigma2
    resid = returns[t] - mu
    log_dens = -0.5 * np.log(2.0 * np.pi * sigma2) - 0.5 * (resid ** 2) / sigma2
    max_ld = np.max(log_dens)
    joint = xi_pred * np.exp(log_dens - max_ld)
    lik_t = joint.sum()
    xi_prev = joint / lik_t if lik_t >= 1e-300 else np.ones(K) / K
    sigma2_prev = sigma2
    xi_filtered[t] = xi_prev

xi_smoothed = np.zeros((T, K))
xi_smoothed[-1] = xi_filtered[-1].copy()
for t in range(T - 2, -1, -1):
    P = transition_matrix(z_path[t + 1], trans_intercept, s_enter, s_exit)
    xi_pred = np.maximum(P.T @ xi_filtered[t], 1e-12)
    xi_smoothed[t] = xi_filtered[t] * (P @ (xi_smoothed[t + 1] / xi_pred))
    xi_smoothed[t] /= xi_smoothed[t].sum()

P_avg = np.zeros((K, K))
for t in range(T):
    P_avg += transition_matrix(z_path[t], trans_intercept, s_enter, s_exit)
P_avg /= T
exp_dur = 1.0 / np.maximum(1.0 - np.diag(P_avg), 1e-8)
pi_stat = stationary_dist(P_avg)

print("\nAverage transition matrix:")
print(np.round(P_avg, 4))
print(f"E[dur] days: Normal={exp_dur[0]:.1f}, Stress={exp_dur[1]:.1f}")
print(f"Stationary mass: Normal={pi_stat[0]:.3f}, Stress={pi_stat[1]:.3f}")
print(f"Smoothed Stress freq: {(np.argmax(xi_smoothed,1)==1).mean():.3f}")
print(f"Mean filtered Stress mass: {xi_filtered[:,1].mean():.3f}")

most_likely = np.argmax(xi_smoothed, axis=1)
rows = []
for k in range(K):
    mask = most_likely == k
    r = returns[mask]
    inv = inversion_lagged[mask]
    lv = logvix_lagged[mask]
    rows.append({
        "Regime": names[k],
        "N_obs": int(mask.sum()),
        "Frequency": float(mask.mean()),
        "Mean_return": float(r.mean()) if mask.any() else np.nan,
        "Std_return": float(r.std()) if mask.any() else np.nan,
        "Mean_logVIX": float(lv.mean()) if mask.any() else np.nan,
        "Mean_Inversion": float(inv.mean()) if mask.any() else np.nan,
    })
char_df = pd.DataFrame(rows).set_index("Regime")
print("\nRegime characteristics (most-likely smoothed):")
print(char_df.round(4))
char_df.to_csv("regime_characteristics_2f.csv")
print("Saved → regime_characteristics_2f.csv")


## 6. Smoothed Regime Probabilities

In [ ]:
# Smoothed regime probabilities + SPY with regime background
# Green = Normal Growth (0), Red/Salmon = High-Vol / Stress (1)
# Background on BOTH panels uses most-likely smoothed state (visual only).

most_likely = np.argmax(xi_smoothed, axis=1)
dates = weekly.index
spy_px = weekly["SPY"].values.astype(float)

fig, axes = plt.subplots(
    2, 1, figsize=(14, 8), sharex=True,
    gridspec_kw={"height_ratios": [1.15, 1.0]},
)

# ----- Panel 1: smoothed probabilities -----
ax = axes[0]
ax.fill_between(
    dates, 0.0, xi_smoothed[:, 0],
    color="green", alpha=0.55, linewidth=0, label="Normal Growth (0)",
)
ax.fill_between(
    dates, xi_smoothed[:, 0], 1.0,
    color="salmon", alpha=0.55, linewidth=0, label="High-Vol / Stress (1)",
)
ax.plot(dates, xi_smoothed[:, 0], color="darkgreen", lw=0.6, alpha=0.8)
ax.set_ylim(0.0, 1.02)
ax.set_ylabel("Smoothed P(regime)")
ax.set_title(
    "Smoothed regime probabilities (asymmetric persistence + Stress-mass penalty)"
)
ax.legend(loc="upper right", framealpha=0.9)

# ----- Panel 2: SPY price with regime background -----
ax = axes[1]

# Shade contiguous regime segments behind the price path
t0 = 0
for t in range(1, T + 1):
    if t == T or most_likely[t] != most_likely[t0]:
        color = "green" if most_likely[t0] == 0 else "salmon"
        ax.axvspan(
            dates[t0], dates[min(t, T - 1)],
            color=color, alpha=0.22, linewidth=0, zorder=0,
        )
        t0 = t

ax.plot(dates, spy_px, color="navy", lw=1.15, zorder=2, label="SPY")
ax.set_ylabel("SPY")
ax.set_title("SPY price with regime background (green = Normal, red = Stress)")
ax.legend(loc="upper left", framealpha=0.9)

# Optional: light grid only on price panel
ax.grid(True, alpha=0.25)

fig.autofmt_xdate()
plt.tight_layout()
plt.savefig("regime_probabilities_2f.png", dpi=150, bbox_inches="tight")
plt.savefig("regime_probabilities_2f.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved -> regime_probabilities_2f.png")
print("Figure saved -> regime_probabilities_2f.pdf")
print(
    f"Smoothed most-likely frequency: "
    f"Normal={(most_likely == 0).mean() * 100:.1f}%  "
    f"Stress={(most_likely == 1).mean() * 100:.1f}%"
)


## 7. Trading Probabilities (EMA)

In [ ]:
# Trading probabilities — EMA smoother (implementation layer)
LAM = 0.85
pi_trade = np.zeros((T, K))
pi_trade[0] = xi_filtered[0].copy()
for t in range(1, T):
    pi_trade[t] = LAM * pi_trade[t-1] + (1.0 - LAM) * xi_filtered[t]
    pi_trade[t] /= pi_trade[t].sum()

print(f"Trading EMA (LAM={LAM}) ready")
print(f"  Mean P(Stress) filtered = {xi_filtered[:,1].mean():.3f}")
print(f"  Mean P(Stress) trade    = {pi_trade[:,1].mean():.3f}")


## 8. Soft Position Mapping

In [ ]:
# Risk-minimization allocation
from scipy.stats import norm as _norm

def mixture_es_gaussian(w, probs, mu_vec, s2_vec, alpha=0.05, n_grid=800):
    w = float(w)
    probs = np.asarray(probs, dtype=float)
    mu_vec = np.asarray(mu_vec, dtype=float)
    s2_vec = np.asarray(s2_vec, dtype=float)
    K_ = len(probs)
    port_mu = w * mu_vec
    port_std = np.abs(w) * np.sqrt(np.maximum(s2_vec, 1e-16))
    left = float(np.min(port_mu - 6.0 * port_std))
    right = float(np.max(port_mu + 2.0 * port_std))
    if not np.isfinite(left) or not np.isfinite(right) or right <= left:
        left, right = -0.30, 0.10
    grid = np.linspace(left, right, n_grid)
    dens = np.zeros(n_grid, dtype=float)
    for k in range(K_):
        dens += probs[k] * _norm.pdf(grid, loc=port_mu[k], scale=max(port_std[k], 1e-12))
    mass = dens.sum()
    if mass < 1e-300:
        return float(port_mu @ probs)
    dens /= mass
    cdf = np.cumsum(dens)
    idx = int(np.searchsorted(cdf, alpha))
    idx = min(max(idx, 1), n_grid - 1)
    tail = dens[: idx + 1]
    if tail.sum() < 1e-300:
        return float(grid[idx])
    return float(np.average(grid[: idx + 1], weights=tail))

USE_TRADE_PI = True
MODE = "soft"
TAU_STRESS = 0.65
W_STRESS   = 0.50
W_NORMAL   = 1.00
USE_ES_GUARD = False
ES_ALPHA = 0.05
ES_FLOOR = -0.025

def target_weight_from_pi(pi, mu_vec, s2_vec):
    p_stress = float(pi[1])
    if MODE == "hard":
        w = W_STRESS if p_stress >= TAU_STRESS else W_NORMAL
    else:
        w = W_NORMAL + (W_STRESS - W_NORMAL) * np.clip(p_stress / max(TAU_STRESS, 1e-6), 0, 1)
    if USE_ES_GUARD:
        es = mixture_es_gaussian(w, pi, mu_vec, s2_vec, alpha=ES_ALPHA)
        if es < ES_FLOOR:
            w = W_STRESS
    return float(np.clip(w, min(W_STRESS, W_NORMAL), max(W_STRESS, W_NORMAL)))

w_target = np.full(T, np.nan)
for t in range(T):
    pi = pi_trade[t] if USE_TRADE_PI else xi_filtered[t]
    w_target[t] = target_weight_from_pi(pi, mu, sigma2_path[t])

print(f"Target weights ready | MODE={MODE} TAU={TAU_STRESS} W_STRESS={W_STRESS}")
print(f"  Mean target weight = {np.nanmean(w_target):.3f}")
print(f"  Fraction in Stress weight = {(w_target <= W_STRESS + 1e-8).mean():.3f}")


## 9. Execution Constraints (Hysteresis + Min-Hold)

In [ ]:
# Asymmetric hysteresis + minimum holding period
MIN_HOLD = 5
BAND_UP   = 0.12
BAND_DOWN = 0.08

w_exec = np.full(T, np.nan)
w_exec[0] = w_target[0]
last_change = 0

for t in range(1, T):
    target = w_target[t]
    current = w_exec[t-1]
    held = t - last_change
    if held < MIN_HOLD:
        w_exec[t] = current
        continue
    if target > current + BAND_UP:
        w_exec[t] = target
        last_change = t
    elif target < current - BAND_DOWN:
        w_exec[t] = target
        last_change = t
    else:
        w_exec[t] = current

print(f"Execution weights ready | MIN_HOLD={MIN_HOLD}  BAND_UP={BAND_UP}  BAND_DOWN={BAND_DOWN}")
print(f"  Mean exec weight = {np.nanmean(w_exec):.3f}")
n_changes = np.sum(np.abs(np.diff(w_exec)) > 1e-8)
print(f"  Number of weight changes = {n_changes}")
print(f"  Approx annual turnover (one-way) ≈ {n_changes * 252 / T:.1f}")


## 10. Strategy Returns (Costs + Stylised Tax)

In [ ]:
# Strategy returns with transaction costs + stylised ST/LT tax
COST_BPS = 2.0
TAX_ST   = 0.37
TAX_LT   = 0.20
TAX_LT_DAYS = 365

spy_ret = weekly["SPY_return"].values.astype(np.float64)
bil_ret = weekly["BIL_return"].values.astype(np.float64)

w_lag = np.roll(w_exec, 1)
w_lag[0] = w_exec[0]

gross = w_lag * spy_ret + (1.0 - w_lag) * bil_ret
dw = np.abs(np.diff(w_exec, prepend=w_exec[0]))
cost = dw * (COST_BPS / 10000.0)
net = gross - cost

tax_drag = np.zeros(T)
last_ch = 0
for t in range(1, T):
    if abs(w_exec[t] - w_exec[t-1]) > 1e-8:
        held = t - last_ch
        equity_pnl = w_exec[t-1] * spy_ret[t]
        if equity_pnl > 0 and w_exec[t] < w_exec[t-1]:
            rate = TAX_LT if held >= TAX_LT_DAYS else TAX_ST
            tax_drag[t] = rate * equity_pnl * (w_exec[t-1] - w_exec[t])
        last_ch = t

net_tax = net - tax_drag

def perf_stats(r):
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 10:
        return {"AnnRet": np.nan, "AnnVol": np.nan, "Sharpe": np.nan,
                "MaxDD": np.nan, "Sortino": np.nan}
    ann = 252.0
    mu = r.mean() * ann
    vol = r.std() * np.sqrt(ann)
    sharpe = mu / vol if vol > 1e-12 else np.nan
    wealth = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(wealth)
    dd = wealth / peak - 1.0
    maxdd = dd.min()
    downside = r[r < 0]
    sortino = mu / (downside.std() * np.sqrt(ann)) if len(downside) > 5 else np.nan
    return {"AnnRet": mu, "AnnVol": vol, "Sharpe": sharpe, "MaxDD": maxdd, "Sortino": sortino}

stats_bh   = perf_stats(spy_ret)
stats_net  = perf_stats(net)
stats_tax  = perf_stats(net_tax)

full_table = pd.DataFrame({
    "BuyHold": stats_bh,
    "Strategy_net": stats_net,
    "Strategy_net_tax": stats_tax,
}).T
print("Full-sample (illustrative) performance")
print(full_table.round(4))
full_table.to_csv("performance_fullsample_2f.csv")
print("Saved → performance_fullsample_2f.csv")
print(f"Mean weight = {np.nanmean(w_exec):.3f}")


## 11. Full-sample Equity Curves (Illustrative)

In [ ]:
# Equity curves and drawdowns (full-sample illustration)
def wealth_dd(r):
    w = np.cumprod(1.0 + np.asarray(r, dtype=float))
    peak = np.maximum.accumulate(w)
    return w, w / peak - 1.0

w_bh, dd_bh = wealth_dd(spy_ret)
w_st, dd_st = wealth_dd(net_tax)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(weekly.index, w_bh, color="gray", lw=1.2, label="Buy & Hold")
axes[0].plot(weekly.index, w_st, color="darkred", lw=1.2, label="Strategy (net+tax)")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()
axes[0].set_title("Full-sample equity (two-factor TVTP, illustrative)")

axes[1].plot(weekly.index, dd_bh, color="gray", lw=1.0, label="Buy & Hold")
axes[1].plot(weekly.index, dd_st, color="darkred", lw=1.0, label="Strategy")
axes[1].set_ylabel("Drawdown")
axes[1].legend()
plt.tight_layout()
plt.savefig("equity_drawdown_fullsample_2f.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → equity_drawdown_fullsample_2f.png")


## 12. Expanding-Window Out-of-Sample Evaluation

In [ ]:
# Strict expanding-window OOS (primary evaluation)
TRAIN_MIN = 1500
REEST_EVERY = 130
N_STARTS_OOS = 6
MAX_ITER_OOS = 250

MODE_OOS = MODE
TAU_STRESS_OOS = TAU_STRESS
W_STRESS_OOS = W_STRESS
W_NORMAL_OOS = W_NORMAL
LAM_OOS = LAM
MIN_HOLD_OOS = MIN_HOLD
BAND_UP_OOS = BAND_UP
BAND_DOWN_OOS = BAND_DOWN
COST_BPS_OOS = COST_BPS
TAX_ST_OOS = TAX_ST
TAX_LT_OOS = TAX_LT
TAX_LT_DAYS_OOS = TAX_LT_DAYS

def negll_segment(theta, ret_seg, z_seg):
    mu, omega, alpha, gamma, beta, t_int, s_enter, s_exit = unpack_parameters(theta)
    if not (omega[0] < omega[1]):
        return 1e12 + 1e6 * max(0.0, omega[0] - omega[1])
    P_all = build_P_all(z_seg, t_int, s_enter, s_exit)
    nll = float(nll_numba(ret_seg, mu, omega, alpha, gamma, beta, P_all))
    P_avg = P_all.mean(axis=0)
    stay0, stay1 = float(P_avg[0,0]), float(P_avg[1,1])
    stress_mass = float(stationary_dist(P_avg)[1])
    TARGET_STAY0, TARGET_STAY1 = 0.97, 0.90
    TARGET_STRESS_MASS, STRESS_MASS_BAND = 0.25, 0.05
    PEN_STAY, PEN_MASS = 200.0, 150.0
    pen_stay = PEN_STAY * (max(0.0, TARGET_STAY0-stay0)**2 + max(0.0, TARGET_STAY1-stay1)**2)
    excess = max(0.0, stress_mass - (TARGET_STRESS_MASS + STRESS_MASS_BAND))
    shortfall = max(0.0, (TARGET_STRESS_MASS - STRESS_MASS_BAND) - stress_mass)
    pen_mass = PEN_MASS * (excess**2 + 0.25*shortfall**2)
    return nll + pen_stay + pen_mass


def estimate_on_segment(ret_seg, z_seg, warm_theta=None, n_starts=N_STARTS_OOS):
    best_local = np.inf
    best_th = None
    starts = []
    if warm_theta is not None:
        starts.append(warm_theta.copy())
    rng = np.random.default_rng(RANDOM_SEED + len(ret_seg))
    for _ in range(n_starts):
        starts.append(random_theta(rng))
    for th0 in starts:
        try:
            res = minimize(
                lambda th: negll_segment(th, ret_seg, z_seg),
                th0, method="L-BFGS-B",
                options={"maxiter": MAX_ITER_OOS, "ftol": 1e-8, "disp": False},
            )
            fun = float(res.fun) if np.isfinite(res.fun) else np.inf
            if fun < best_local:
                best_local = fun
                best_th = res.x.copy()
        except Exception:
            continue
    return best_th, best_local


def filter_segment(ret_seg, z_seg, theta):
    mu_e, omega_e, alpha_e, gamma_e, beta_e, t_int, s_enter, s_exit = unpack_parameters(theta)
    T_ = len(ret_seg)
    xi = np.zeros((T_, K))
    s2_path = np.zeros((T_, K))
    xi_prev = np.ones(K) / K
    s2_prev = np.full(K, np.var(ret_seg[:min(50, T_)]))
    for t in range(T_):
        P = transition_matrix(z_seg[t], t_int, s_enter, s_exit)
        xi_pred = P.T @ xi_prev
        if t > 0:
            eps = ret_seg[t-1] - mu_e
            Ineg = (eps < 0).astype(np.float64)
        else:
            eps = np.zeros(K)
            Ineg = np.zeros(K)
        s2 = omega_e + (alpha_e + gamma_e * Ineg) * (eps**2) + beta_e * s2_prev
        s2 = np.maximum(s2, 1e-12)
        s2_path[t] = s2
        resid = ret_seg[t] - mu_e
        log_dens = -0.5*np.log(2*np.pi*s2) - 0.5*(resid**2)/s2
        max_ld = np.max(log_dens)
        joint = xi_pred * np.exp(log_dens - max_ld)
        lik = joint.sum()
        xi_prev = joint / lik if lik >= 1e-300 else np.ones(K)/K
        s2_prev = s2
        xi[t] = xi_prev
    return xi, s2_path, (mu_e, omega_e, alpha_e, gamma_e, beta_e)


z_full = np.column_stack([logvix_lagged, inversion_lagged])
reest_dates = list(range(TRAIN_MIN, T, REEST_EVERY))
if not reest_dates:
    raise ValueError("TRAIN_MIN too large for sample")

print(f"Expanding OOS setup")
print(f"  T = {T}  |  first OOS from day {TRAIN_MIN}")
print(f"  Re-estimate every {REEST_EVERY} days  |  n re-estimations = {len(reest_dates)}")
print(f"  Starts per re-est = {N_STARTS_OOS} + warm-start")
print("=" * 72)

w_oos = np.full(T, np.nan)
w_oos_exec = np.full(T, np.nan)
warm = best_theta.copy()
t0_all = time.time()

for i, t_reest in enumerate(reest_dates):
    ret_seg = returns[:t_reest]
    z_seg   = z_full[:t_reest]
    t_est0 = time.time()
    theta_hat, nll_hat = estimate_on_segment(ret_seg, z_seg, warm_theta=warm, n_starts=N_STARTS_OOS)
    dt_est = time.time() - t_est0
    if theta_hat is None:
        print(f"[{i+1}/{len(reest_dates)}] t={t_reest}: estimation failed — keep previous")
        theta_hat = warm
        nll_hat = np.nan
    else:
        warm = theta_hat.copy()

    t_end = reest_dates[i+1] if (i+1) < len(reest_dates) else T
    xi_f, s2_path, unpacked = filter_segment(returns[:t_end], z_full[:t_end], theta_hat)
    mu_e = unpacked[0]

    if t_reest > 0:
        pi = xi_f[t_reest-1].copy()
    else:
        pi = np.ones(K)/K

    for t in range(t_reest, t_end):
        pi = LAM_OOS * pi + (1.0 - LAM_OOS) * xi_f[t]
        pi = pi / pi.sum()
        w_oos[t] = target_weight_from_pi(pi, mu_e, s2_path[t])

    print(f"[{i+1:02d}/{len(reest_dates)}] reest @ {t_reest:4d} → {t_end-1:4d}  "
          f"| NLL+pen={nll_hat:10.2f}  ({dt_est:.1f}s)")

print("=" * 72)
print(f"All re-estimations done in {(time.time()-t0_all)/60:.1f} min")

first = TRAIN_MIN
w_oos_exec[first] = w_oos[first]
last_change = first
for t in range(first+1, T):
    if not np.isfinite(w_oos[t]):
        w_oos_exec[t] = w_oos_exec[t-1]
        continue
    target = w_oos[t]
    current = w_oos_exec[t-1]
    held = t - last_change
    if held < MIN_HOLD_OOS:
        w_oos_exec[t] = current
        continue
    if target > current + BAND_UP_OOS:
        w_oos_exec[t] = target
        last_change = t
    elif target < current - BAND_DOWN_OOS:
        w_oos_exec[t] = target
        last_change = t
    else:
        w_oos_exec[t] = current

w_lag = np.roll(w_oos_exec, 1)
w_lag[first] = w_oos_exec[first]
gross = w_lag * spy_ret + (1.0 - w_lag) * bil_ret
dw = np.abs(np.diff(w_oos_exec, prepend=w_oos_exec[0]))
cost = dw * (COST_BPS_OOS / 10000.0)
net = gross - cost

tax_drag = np.zeros(T)
last_ch = first
for t in range(first+1, T):
    if abs(w_oos_exec[t] - w_oos_exec[t-1]) > 1e-8:
        held = t - last_ch
        equity_pnl = w_oos_exec[t-1] * spy_ret[t]
        if equity_pnl > 0 and w_oos_exec[t] < w_oos_exec[t-1]:
            rate = TAX_LT_OOS if held >= TAX_LT_DAYS_OOS else TAX_ST_OOS
            tax_drag[t] = rate * equity_pnl * (w_oos_exec[t-1] - w_oos_exec[t])
        last_ch = t
net_tax = net - tax_drag

oos_mask = (np.arange(T) >= first) & np.isfinite(net_tax) & np.isfinite(spy_ret)
stats_st = perf_stats(net_tax[oos_mask])
stats_bh = perf_stats(spy_ret[oos_mask])
stats_net = perf_stats(net[oos_mask])
stats_gr = perf_stats(gross[oos_mask])

oos_table = pd.DataFrame({
    "BuyHold_OOS": stats_bh,
    "Strategy_gross_OOS": stats_gr,
    "Strategy_net_OOS": stats_net,
    "Strategy_net_tax_OOS": stats_st,
}).T
print("\nStrict expanding-window OOS performance (two-factor)")
print(oos_table.round(4))
oos_table.to_csv("performance_oos_expanding_2f.csv")
print("Saved → performance_oos_expanding_2f.csv")

turnover_oos = float(np.nansum(np.abs(np.diff(w_oos_exec[first:]))) * (252.0 / max(T-first, 1)))
print(f"OOS annual turnover ≈ {turnover_oos:.2f}")
print(f"OOS mean weight ≈ {np.nanmean(w_oos_exec[first:]):.3f}")

w_bh, dd_bh = wealth_dd(spy_ret[oos_mask])
w_st, dd_st = wealth_dd(net_tax[oos_mask])
idx_oos = weekly.index[oos_mask]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(idx_oos, w_bh, color="gray", lw=1.2, label="Buy & Hold")
axes[0].plot(idx_oos, w_st, color="darkred", lw=1.2, label="Strategy (net+tax)")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()
axes[0].set_title("Expanding-window OOS equity (two-factor, re-estimated)")

axes[1].plot(idx_oos, dd_bh, color="gray", lw=1.0, label="Buy & Hold")
axes[1].plot(idx_oos, dd_st, color="darkred", lw=1.0, label="Strategy")
axes[1].set_ylabel("Drawdown")
axes[1].legend()
plt.tight_layout()
plt.savefig("equity_drawdown_oos_expanding_2f.png", dpi=150, bbox_inches="tight")
plt.savefig("equity_drawdown_oos_expanding_2f.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → equity_drawdown_oos_expanding_2f.png")
print("Figure saved → equity_drawdown_oos_expanding_2f.pdf")


## 13. Episode Diagnosis

In [ ]:
# Episode diagnosis (drawdowns vs VIX complex under two-factor)
px = weekly["SPY"].astype(float)
ret = weekly["SPY_return"].astype(float)
inv = weekly["Inversion"].astype(float)
inv_lag = weekly["Inversion_lagged"].astype(float)
vix = weekly["VIX"].astype(float)
vix3m = weekly["VIX3M"].astype(float)
p_stress = xi_smoothed[:, 1]

wealth = (1.0 + ret).cumprod()
peak = wealth.cummax()
dd = wealth / peak - 1.0

EPISODES = [
    ("EU_2011",      "2011-07-01", "2011-10-15"),
    ("China_2015",   "2015-07-01", "2015-09-30"),
    ("Volmageddon",  "2018-01-15", "2018-04-15"),
    ("COVID_2020",   "2020-02-01", "2020-05-01"),
    ("Bear_2022",    "2022-01-01", "2022-10-31"),
    ("Aug_2024",     "2024-07-15", "2024-09-15"),
]

def slice_episode(start, end):
    return (weekly.index >= start) & (weekly.index <= end)

rows = []
for name, start, end in EPISODES:
    m = slice_episode(start, end)
    if m.sum() < 5:
        continue
    dd_ep = dd[m]
    rows.append({
        "Episode": name,
        "Start": start,
        "End": end,
        "MinDD": float(dd_ep.min()),
        "Max_Inversion": float(inv[m].max()),
        "Mean_Inversion": float(inv[m].mean()),
        "Frac_Inv_gt1": float((inv[m] > 1.0).mean()),
        "Max_P_Stress": float(p_stress[m].max()),
        "Mean_P_Stress": float(p_stress[m].mean()),
        "Frac_P_gt05": float((p_stress[m] > 0.5).mean()),
        "Max_VIX": float(vix[m].max()),
        "Max_VIX3M": float(vix3m[m].max()),
    })

epi_df = pd.DataFrame(rows).set_index("Episode")
print("Episode detection table (two-factor smoothed P)")
print(epi_df.round(3).to_string())
epi_df.to_csv("episode_diagnosis_2f.csv")
print("\nSaved → episode_diagnosis_2f.csv")

print("\nIllustrative under-detection flags (MinDD<-10%, Frac_Inv>1<25%, Mean_P<0.35):")
for ep, row in epi_df.iterrows():
    flag = (row["MinDD"] < -0.10) and (row["Frac_Inv_gt1"] < 0.25) and (row["Mean_P_Stress"] < 0.35)
    print(f"  {ep}: {'FLAG' if flag else 'ok'}  "
          f"MinDD={row['MinDD']:.1%}  FracInv>1={row['Frac_Inv_gt1']:.1%}  "
          f"MeanP={row['Mean_P_Stress']:.2f}")


## 14. Signal Alignment Diagnostics

In [ ]:
# Inversion / P(Stress) alignment when SPY already down >5%
in_dd = dd < -0.05
inv_dd = inv_lag[in_dd]
p_dd   = p_stress[in_dd]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(inv_dd, p_dd, alpha=0.25, s=12, c="steelblue")
ax.axvline(1.0, color="orange", ls="--", lw=1, label="Inversion=1")
ax.axhline(0.5, color="crimson", ls="--", lw=1, label="P(Stress)=0.5")
ax.set_xlabel("Inversion (lagged)")
ax.set_ylabel("P(Stress) smoothed")
ax.set_title("When SPY DD < -5%: Inversion vs P(Stress) [two-factor]")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("inversion_vs_pstress_in_drawdown_2f.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → inversion_vs_pstress_in_drawdown_2f.png")


## 15. Summary Statistics

In [ ]:
# Quick summary print
print("=" * 72)
print("TWO-FACTOR TVTP-MS-GJR-GARCH — RUN SUMMARY")
print("=" * 72)
print(f"Sample: {weekly.index.min().date()} → {weekly.index.max().date()}  (T={T})")
print(f"Features: log(VIX), Inversion  — both lagged 1 day")
print(f"s_enter (>=0): {np.round(s_enter, 4)}")
print(f"s_exit  (free): {np.round(s_exit, 4)}")
print(f"Stationary Stress mass ≈ {pi_stat[1]:.3f}")
print(f"E[dur] Normal / Stress (days): {exp_dur[0]:.1f} / {exp_dur[1]:.1f}")
print()
print("Primary evaluation is the expanding OOS table (Cell 12).")
print("Full-sample equity is illustrative only.")
print("Episode table (Cell 13) documents coverage of major stress windows.")
print("=" * 72)


## 16. Soft-Mapping Function Visualisation

In [ ]:
# Soft-mapping function (for visualisation Figure)

tau = 0.65
w_stress = 0.50
p_grid = np.linspace(0.0, 1.0, 500)
w_grid = 1.0 + (w_stress - 1.0) * np.clip(p_grid / tau, 0.0, 1.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_grid, w_grid, color="darkblue", lw=2.4,
        label=r"$w=1+(W_{\mathrm{STRESS}}-1)\cdot\mathrm{clip}(P/\tau,0,1)$")
ax.axvline(tau, color="crimson", ls="--", lw=1.3, label=fr"$\tau={tau}$")
ax.axhline(w_stress, color="gray", ls=":", lw=1.1)
ax.axhline(1.0, color="gray", ls=":", lw=1.1)
ax.set_xlabel("Filtered P(Stress)")
ax.set_ylabel("Equity weight $w$")
ax.set_title("Soft-mapping function (frozen calibration)")
ax.set_xlim(0, 1)
ax.set_ylim(0.45, 1.05)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("fig_softmap.pdf", dpi=200, bbox_inches="tight")
plt.savefig("fig_softmap.png", dpi=200, bbox_inches="tight")
plt.show()
print("Figure saved → fig_softmap.pdf / fig_softmap.png")


## 17. Episode Signal Plots (2015 / 2022)

In [ ]:
# 2015 & 2022 episode signals (for visualisation Figure)
# Requires from earlier cells:
#   weekly (DataFrame with DatetimeIndex)
#   p_stress (array, smoothed Stress probability, length T)
#   inversion_lagged or weekly["Inversion_lagged"]
# Optional: dlog_VIX_lagged (computed on the fly if missing)

inv = weekly["Inversion_lagged"].values.astype(float)

if "dlog_VIX_lagged" in weekly.columns:
    dlog = weekly["dlog_VIX_lagged"].values.astype(float)
else:
    # reconstruct from log_VIX_lagged
    dlog = np.full(len(weekly), np.nan)
    logv = weekly["log_VIX_lagged"].values.astype(float)
    dlog[1:] = np.diff(logv)

idx = weekly.index
episodes = [
    ("China 2015", "2015-07-01", "2015-09-30"),
    ("Bear 2022",  "2022-01-01", "2022-10-31"),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex="row")

for row, (name, start, end) in enumerate(episodes):
    mask = (idx >= start) & (idx <= end)
    t = idx[mask]

    axes[row, 0].plot(t, inv[mask], color="steelblue", lw=1.1)
    axes[row, 0].axhline(1.0, color="orange", ls="--", lw=1.0)
    axes[row, 0].set_ylabel("Inversion")
    axes[row, 0].set_title(f"{name}: Inversion")

    axes[row, 1].plot(t, dlog[mask], color="darkgreen", lw=1.1)
    axes[row, 1].axhline(0.0, color="gray", ls="--", lw=0.9)
    axes[row, 1].set_title(f"{name}: Δlog(VIX)")

    axes[row, 2].plot(t, p_stress[mask], color="crimson", lw=1.2)
    axes[row, 2].axhline(0.5, color="gray", ls="--", lw=0.9)
    axes[row, 2].set_ylim(-0.02, 1.05)
    axes[row, 2].set_title(f"{name}: P(Stress)")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Episode signals: 2015 China vs 2022 Bear (two-factor)", y=1.01)
plt.tight_layout()
plt.savefig("fig_episode_2015_2022.pdf", dpi=200, bbox_inches="tight")
plt.savefig("fig_episode_2015_2022.png", dpi=200, bbox_inches="tight")
plt.show()
print("Figure saved → fig_episode_2015_2022.pdf / fig_episode_2015_2022.png")


## 18. Out-of-Sample Weight Path

In [ ]:
# OOS equity weight path (for visualisation Figure)
# Requires from Cell 12 (expanding OOS):
#   w_oos_exec  (array of executed weights)
#   first       (integer index where OOS begins)
#   weekly.index

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(weekly.index[first:], w_oos_exec[first:], color="darkblue", lw=0.95)
ax.axhline(np.nanmean(w_oos_exec[first:]), color="gray", ls="--", lw=1.2,
           label=f"Mean w ≈ {np.nanmean(w_oos_exec[first:]):.3f}")
ax.axhline(0.50, color="crimson", ls=":", lw=1.1, label=r"$W_{\mathrm{STRESS}}=0.50$")
ax.set_ylabel("Equity weight $w_t$")
ax.set_title("Expanding-window OOS equity weight path (frozen soft map, min-hold=5)")
ax.set_ylim(0.45, 1.05)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("fig_weight_path.pdf", dpi=200, bbox_inches="tight")
plt.savefig("fig_weight_path.png", dpi=200, bbox_inches="tight")
plt.show()
print("Figure saved → fig_weight_path.pdf / fig_weight_path.png")


## 19. Tail Metrics and Practitioner Benchmarks

In [ ]:
# Tail metrics (CVaR / Downside Dev) + Practitioner Benchmarks
# Paste AFTER the expanding-window OOS cell (Cell 12).
# Requires: net_tax, spy_ret, bil_ret, oos_mask, first, weekly, T,
#           COST_BPS_OOS, TAX_ST_OOS, TAX_LT_OOS, TAX_LT_DAYS_OOS
# All comments in English as required.

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Helper: performance + tail metrics
# ---------------------------------------------------------------------------
def perf_stats_full(r, ann=252.0):
    """Annualized return/vol/Sharpe/MaxDD/Calmar + CVaR95 + downside deviation."""
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]
    out = {k: np.nan for k in [
        "AnnRet", "AnnVol", "Sharpe", "MaxDD", "Calmar",
        "CVaR95", "DownsideDev", "MeanW"
    ]}
    if len(r) < 30:
        return out

    mu = r.mean() * ann
    vol = r.std(ddof=1) * np.sqrt(ann)
    sharpe = mu / vol if vol > 1e-12 else np.nan

    wealth = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(wealth)
    dd = wealth / peak - 1.0
    maxdd = float(dd.min())
    calmar = mu / abs(maxdd) if abs(maxdd) > 1e-12 else np.nan

    # 95% CVaR (Expected Shortfall): mean of returns at or below 5% quantile
    q = np.quantile(r, 0.05)
    tail = r[r <= q]
    cvar = float(tail.mean()) if len(tail) > 0 else np.nan
    # Report as positive loss number (common practitioner convention)
    cvar_loss = -cvar * ann  # annualized approximate loss magnitude (optional scale)
    # Keep daily-mean CVaR as well for the table; paper can use either
    cvar_daily = float(cvar)  # negative number

    downside = r[r < 0.0]
    ddev = float(downside.std(ddof=1) * np.sqrt(ann)) if len(downside) > 5 else np.nan

    out.update({
        "AnnRet": mu,
        "AnnVol": vol,
        "Sharpe": sharpe,
        "MaxDD": maxdd,
        "Calmar": calmar,
        "CVaR95": cvar_daily,       # daily expected shortfall (negative)
        "DownsideDev": ddev,
    })
    return out


def apply_costs_and_tax(w, spy, bil, first_idx, cost_bps, tax_st, tax_lt, tax_lt_days):
    """Replicate the paper's cost + stylised ST/LT tax drag on a weight path."""
    w = np.asarray(w, dtype=float)
    Tloc = len(w)
    w_lag = np.roll(w, 1)
    w_lag[first_idx] = w[first_idx]
    gross = w_lag * spy + (1.0 - w_lag) * bil
    dw = np.abs(np.diff(w, prepend=w[0]))
    cost = dw * (cost_bps / 10000.0)
    net = gross - cost

    tax_drag = np.zeros(Tloc)
    last_ch = first_idx
    for t in range(first_idx + 1, Tloc):
        if abs(w[t] - w[t - 1]) > 1e-8:
            held = t - last_ch
            equity_pnl = w[t - 1] * spy[t]
            if equity_pnl > 0 and w[t] < w[t - 1]:
                rate = tax_lt if held >= tax_lt_days else tax_st
                tax_drag[t] = rate * equity_pnl * (w[t - 1] - w[t])
            last_ch = t
    return net - tax_drag


# ---------------------------------------------------------------------------
# 2. Tail metrics for locked 2F strategy + Buy & Hold (same OOS window)
# ---------------------------------------------------------------------------
r_st = net_tax[oos_mask]
r_bh = spy_ret[oos_mask]

st_tail = perf_stats_full(r_st)
bh_tail = perf_stats_full(r_bh)
st_tail["MeanW"] = float(np.nanmean(w_oos_exec[first:]))

print("=" * 72)
print("TAIL METRICS (locked 2F soft OOS vs Buy & Hold)")
print("=" * 72)
tail_df = pd.DataFrame({"2F_soft_Main": st_tail, "BuyHold": bh_tail}).T
print(tail_df[["AnnRet", "AnnVol", "MaxDD", "Sharpe", "Calmar", "CVaR95", "DownsideDev"]].round(4))
tail_df.to_csv("tail_metrics_oos_2f.csv")
print("Saved → tail_metrics_oos_2f.csv")
print()
print("Key numbers (Tail-Risk Metrics):")
print(f"  2F soft  Calmar={st_tail['Calmar']:.2f}  "
      f"CVaR95={st_tail['CVaR95']:.4f}  DownsideDev={st_tail['DownsideDev']:.4f}")
print(f"  BuyHold  Calmar={bh_tail['Calmar']:.2f}  "
      f"CVaR95={bh_tail['CVaR95']:.4f}  DownsideDev={bh_tail['DownsideDev']:.4f}")


# ---------------------------------------------------------------------------
# 3. Practitioner benchmarks on the SAME expanding OOS window
# ---------------------------------------------------------------------------
# Design choices (simple, retail-feasible, no look-ahead):
#   (A) Inverse-Vol: w_t = clip( target_vol / realized_vol_{t-1}, 0, 1 )
#       realized_vol = 21-day trailing std * sqrt(252), lagged 1 day
#       target_vol   = 0.12  (matches approx strategy vol level)
#   (B) 200-day MA:  w = 1 if SPY > MA200 (lagged), else 0
#   (C) VIX > 25:    w = 0.50 if lagged VIX > 25, else 1.00
# Costs + stylised tax applied exactly as in the main strategy.

px = weekly["SPY"].astype(float).values
vix_lvl = weekly["VIX"].astype(float).values

# --- (A) Inverse-volatility scaling ---
ret_for_vol = pd.Series(spy_ret, index=weekly.index)
realized_vol = ret_for_vol.rolling(21).std() * np.sqrt(252.0)
realized_vol_lag = realized_vol.shift(1).values  # no look-ahead
TARGET_VOL = 0.12
w_inv = np.clip(TARGET_VOL / np.maximum(realized_vol_lag, 1e-6), 0.0, 1.0)
w_inv[:first] = np.nan  # only evaluate OOS

# --- (B) 200-day moving-average timing ---
ma200 = pd.Series(px, index=weekly.index).rolling(200).mean().shift(1).values
w_ma = np.where(px > ma200, 1.0, 0.0).astype(float)
w_ma[:first] = np.nan

# --- (C) Naive VIX-level threshold ---
vix_lag = pd.Series(vix_lvl, index=weekly.index).shift(1).values
w_vix = np.where(vix_lag > 25.0, 0.50, 1.00).astype(float)
w_vix[:first] = np.nan

benchmarks = {
    "InverseVol": w_inv,
    "MA200": w_ma,
    "VIX_gt25": w_vix,
}

rows = []
for name, w in benchmarks.items():
    # Fill leading NaNs inside OOS with first valid weight (stable start)
    w_fill = w.copy()
    # Use only OOS segment for cost/tax path; keep pre-OOS at 1.0 for lag continuity
    w_path = np.ones(T)
    valid = np.isfinite(w_fill)
    w_path[valid] = w_fill[valid]
    # From first OOS day onward, if still nan, carry forward
    for t in range(first, T):
        if not np.isfinite(w_fill[t]):
            w_path[t] = w_path[t - 1]
        else:
            w_path[t] = w_fill[t]

    r_net = apply_costs_and_tax(
        w_path, spy_ret, bil_ret, first,
        COST_BPS_OOS, TAX_ST_OOS, TAX_LT_OOS, TAX_LT_DAYS_OOS
    )
    stats = perf_stats_full(r_net[oos_mask])
    stats["MeanW"] = float(np.nanmean(w_path[first:]))
    to = float(np.nansum(np.abs(np.diff(w_path[first:]))) * (252.0 / max(T - first, 1)))
    stats["TO"] = to
    rows.append(stats)
    print(f"{name}: MeanW={stats['MeanW']:.3f}  AnnRet={stats['AnnRet']:.2%}  "
          f"AnnVol={stats['AnnVol']:.2%}  MaxDD={stats['MaxDD']:.2%}  "
          f"Sharpe={stats['Sharpe']:.2f}  Calmar={stats['Calmar']:.2f}  "
          f"CVaR95={stats['CVaR95']:.4f}  DD={stats['DownsideDev']:.4f}")

bench_df = pd.DataFrame(rows, index=list(benchmarks.keys()))
# Also attach main strategy + BH for one clean table
main_row = st_tail.copy()
main_row["TO"] = float(np.nansum(np.abs(np.diff(w_oos_exec[first:]))) * (252.0 / max(T - first, 1)))
bh_row = bh_tail.copy()
bh_row["MeanW"] = 1.0
bh_row["TO"] = 0.0

summary = pd.concat([
    pd.DataFrame([main_row], index=["2F_soft_Main"]),
    bench_df,
    pd.DataFrame([bh_row], index=["BuyHold"]),
])
cols = ["MeanW", "AnnRet", "AnnVol", "MaxDD", "Sharpe", "Calmar", "CVaR95", "DownsideDev", "TO"]
print("\n" + "=" * 72)
print("BENCHMARK SUMMARY (same expanding OOS window, net of cost+tax)")
print("=" * 72)
print(summary[cols].round(4).to_string())
summary[cols].to_csv("benchmarks_oos_expanding.csv")
print("\nSaved → benchmarks_oos_expanding.csv")
print("\nTables saved.")
